Estrutura sugerida para o notebook

📌 Célula 1 – Importar bibliotecas

In [ ]:
# Importação da biblioteca para manipulação de dados em tabelas
import pandas as pd  

# Importação da biblioteca NumPy para operações matemáticas e arrays
import numpy as np  

# Importação da biblioteca Matplotlib para geração de gráficos
import matplotlib.pyplot as plt  

# Importação da biblioteca Seaborn para visualização estatística de dados
import seaborn as sns  

# Importação da biblioteca random para geração de números aleatórios
import random  

# Importação das classes datetime e timedelta para manipulação de datas e intervalos de tempo
from datetime import datetime, timedelta  

# Comando mágico do Jupyter Notebook que permite exibir gráficos diretamente no notebook
%matplotlib inline  
import os
import zipfile
import glob
import unidecode


In [ ]:
# Instala o pacote watermark
!pip install -q -U watermark

In [ ]:
%reload_ext watermark
%watermark -a "Luis Antonio" 

In [ ]:
%watermark --iversions

📌 Célula 2 – Extrair todos os CSVs dos zips

In [ ]:
# Caminho base onde estão os arquivos zip
caminho_base = r"C:\Users\User\Documents\CursoDSA\99_analise_dados_reais\202501_PeDeMeia"

# Pasta destino para extrair os CSVs
destino = os.path.join(caminho_base, "csv_extraidos")
os.makedirs(destino, exist_ok=True)

# Lista dos arquivos zip (de janeiro a novembro)
arquivos_zip = [
    "202501_PeDeMeia.zip",
    "202502_PeDeMeia.zip",
    "202503_PeDeMeia.zip",
    "202504_PeDeMeia.zip",
    "202505_PeDeMeia.zip",
    "202506_PeDeMeia.zip",
    "202507_PeDeMeia.zip",
    "202508_PeDeMeia.zip",
    "202509_PeDeMeia.zip",
    "202510_PeDeMeia.zip",
    "202511_PeDeMeia.zip"
]

# Extrair todos os CSVs para a pasta destino
for zip_name in arquivos_zip:
    zip_path = os.path.join(caminho_base, zip_name)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        for nome in zip_ref.namelist():
            if nome.endswith(".csv"):
                zip_ref.extract(nome, destino)

print("Extração concluída. Os CSVs estão na pasta:", destino)


In [ ]:
caminho_base = r"C:\Users\User\Documents\CursoDSA\99_analise_dados_reais\202501_PeDeMeia"
destino = os.path.join(caminho_base, "csv_extraidos")

os.makedirs(destino, exist_ok=True)

arquivos_zip = glob.glob(os.path.join(caminho_base, "*.zip"))

# Evita reextrair se já existir
if not os.listdir(destino):
    for zip_path in arquivos_zip:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(destino)

print("Extração concluída.")

📌 Célula 3 – Ler e concatenar os CSVs

In [ ]:
# Lista todos os CSVs extraídos
arquivos_csv = glob.glob(os.path.join(destino, "*.csv"))

# Lê e concatena todos os CSVs
df_final = pd.concat(
    [pd.read_csv(arq, sep=";", encoding="latin1") for arq in arquivos_csv],
    ignore_index=True
)

print("Shape final:", df_final.shape)
df_final.head()


📌 Célula 4 – Criar colunas auxiliares (ano, mês)

In [ ]:
# Converter coluna "MÊS FOLHA" para datetime
df_final['ano_mes'] = pd.to_datetime(df_final['MÊS FOLHA'], format='%Y%m')
df_final['ano'] = df_final['ano_mes'].dt.year
df_final['mes'] = df_final['ano_mes'].dt.month

df_final[['MÊS FOLHA','ano','mes']].head()


📌 Célula 5 – Exportar consolidado

In [ ]:
# Salvar em CSV final
saida_csv = os.path.join(caminho_base, "pe_de_meia_final.csv")
df_final.to_csv(saida_csv, index=False, encoding="utf-8")

print("Arquivo final salvo em:", saida_csv)


📌 6. Limpeza e Padronização



**Normalização das Colunas**


In [ ]:
# Normalizar nomes de colunas
df_final.columns = (
    df_final.columns
    .str.normalize('NFKD')  # remove acentos
    .str.encode('ascii', errors='ignore')
    .str.decode('utf-8')
    .str.replace(' ', '_')  # troca espaço por underscore
    .str.upper()            # deixa tudo maiúsculo
)

print(df_final.columns)


🔹 Converter valores

In [ ]:
df_final['VALOR_PARCELA'] = (
    df_final['VALOR_PARCELA']
    .astype(str)
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
    .astype(float)
)

🔹 Criar colunas de data

In [ ]:
df_final['ANO_MES'] = pd.to_datetime(df_final['MES_FOLHA'], format='%Y%m')
df_final['ANO'] = df_final['ANO_MES'].dt.year
df_final['MES'] = df_final['ANO_MES'].dt.month

📌 7. Análises


📊 Top Estados

In [ ]:
top_estados = (
    df_final.groupby("UF")["VALOR_PARCELA"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

🏙️ Top Municípios

In [ ]:
top_municipios = (
    df_final.groupby("NOME_MUNICIPIO")["VALOR_PARCELA"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

📈 Evolução Mensal

In [ ]:
evolucao_mensal = (
    df_final.groupby("ANO_MES")["VALOR_PARCELA"]
    .sum()
)

📌 8. Visualizações

📊 Estados

In [ ]:
plt.figure(figsize=(10,5))
top_estados.plot(kind='bar')
plt.title("Top 10 Estados por Valor")
plt.ylabel("Total")
plt.show()

📈 Evolução no tempo

In [ ]:
plt.figure(figsize=(10,5))
evolucao_mensal.plot()
plt.title("Evolução Mensal dos Pagamentos")
plt.ylabel("Total")
plt.show()

📌 9. Exportação

In [ ]:
saida = os.path.join(caminho_base, "pe_de_meia_tratado.csv")
df_final.to_csv(saida, index=False, encoding="utf-8")

print("Arquivo salvo:", saida)